# VALID: how much CIDEr is missed by reranking?

Accelerator **None**, Internet ON. No input Dataset, checkpoint, API token or GPU needed.
One notebook covers CONTROL180 and PROMPT60 END candidates: same5000VALID images; no TEST.
Reuses all saved D/C/O scores and captions; computes CIDEr of each of5ranks with same reference corpus.
Historical rank0 and selected per-image CIDEr must replay before reporting oracle.
Oracle uses references: diagnostic upper bound, not an inference method or visual truth.
One-hour CPU STOPcap, not ETA; Java8/PTB only, no SPICE/CoreNLP parser downloads.
Send `valid_candidate_oracle_v1_review.zip`. No training, decoding or selector weight changes.


In [ ]:
import os,sys,json,time,hashlib,urllib.request,subprocess,threading,queue,signal,tempfile,zipfile,traceback,tarfile
from pathlib import Path
REVISION='590be700ec6ef5fc4b9ae4b705b0c56250b1209c'
FILES={'experiments/csa_paper/score_valid_candidate_oracle.py': '453ce952a0f2970cff6640d2f2ae48e1ce9da88faa4ab168ff49e2fbf607b37a', 'experiments/csa_paper/valid_rerank_diagnostic/VALID_CANDIDATES.json.gz': '05efaac0660f030f892ab1fb4f8bc4d530862b378b109dd187caff85b16abbae'}
OUTPUT=Path('/kaggle/working/valid_candidate_oracle_v1')
if OUTPUT.exists():raise RuntimeError('Preserve old output; use a fresh output/runtime.')
OUTPUT.mkdir(parents=True)
WORK=Path(tempfile.mkdtemp(prefix='valid-oracle-',dir='/tmp'));DEPS=WORK/'deps';LOG=OUTPUT/'notebook.log'
DEADLINE=time.monotonic()+3600
ENV=os.environ.copy()
for k in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):ENV.pop(k,None)
ENV.update(PYTHONPATH=str(DEPS),PYTHONUNBUFFERED='1',CUDA_VISIBLE_DEVICES='',OMP_NUM_THREADS='2')
def run(args,timeout=900):
    start=time.monotonic();beat=start;q=queue.Queue()
    process=subprocess.Popen(list(map(str,args)),env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
    def reader():
        for line in process.stdout:q.put(line)
        q.put(None)
    threading.Thread(target=reader,daemon=True).start()
    try:
        with LOG.open('a',encoding='utf-8') as log:
            while True:
                try:
                    line=q.get(timeout=1)
                    if line is None:break
                    print(line,end='',flush=True);log.write(line);log.flush()
                except queue.Empty:pass
                now=time.monotonic()
                if now>DEADLINE or now-start>timeout:raise TimeoutError('CPU phase/parent cap exceeded')
                if now-beat>=30:print('VALID CPU metric active;',round(now-start),'s; no model.',flush=True);beat=now
        if process.wait():raise subprocess.CalledProcessError(process.returncode,args)
    finally:
        if process.poll() is None:
            os.killpg(process.pid,signal.SIGTERM)
            try:process.wait(timeout=5)
            except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait()


In [ ]:
complete=False
try:
    for name,digest in FILES.items():
        print('Downloading saved source/data:',Path(name).name,flush=True)
        raw=urllib.request.urlopen('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+name,timeout=90).read()
        if hashlib.sha256(raw).hexdigest()!=digest:raise ValueError('Changed pinned file: '+name)
        (WORK/Path(name).name).write_bytes(raw)
    (OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=FILES),indent=2))
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'pycocoevalcap==1.2'],600)
    url='https://github.com/adoptium/temurin8-binaries/releases/download/jdk8u472-b08/OpenJDK8U-jre_x64_linux_hotspot_8u472b08.tar.gz'
    print('Downloading pinned Java8 for PTB tokenizer; no CoreNLP/SPICE models.',flush=True)
    raw=urllib.request.urlopen(url,timeout=120).read()
    if hashlib.sha256(raw).hexdigest()!='6f7fb5fd640a0fd00837344b0920cbc4b9b9284b50e66f33789e3b250446a16e':raise ValueError('Changed Java asset')
    archive=WORK/'java8.tar.gz';archive.write_bytes(raw)
    with tarfile.open(archive) as z:z.extractall(WORK/'jre',filter='data')
    java=list((WORK/'jre').rglob('bin/java'))
    if len(java)!=1:raise ValueError('Missing Java executable')
    ENV['PATH']=str(java[0].parent)+os.pathsep+ENV['PATH']
    run([java[0],'-version'],60)
    bundle_name='experiments/csa_paper/valid_rerank_diagnostic/VALID_CANDIDATES.json.gz'
    run([sys.executable,'-u',WORK/'score_valid_candidate_oracle.py','--bundle',WORK/'VALID_CANDIDATES.json.gz','--sha256',FILES[bundle_name],'--output',OUTPUT],2400)
    complete=json.loads((OUTPUT/'result.json').read_text())['status']=='VALID_CANDIDATE_ORACLE_COMPLETE'
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc());raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,training=False,decoding=False,test_used=False,selector_changed=False)))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.iterdir():
            if p.is_file():z.write(p,p.name)
    print('SEND REVIEW:',review,flush=True)
    print('COMPLETE' if complete else 'INCOMPLETE; preserve outputs; no retraining.',flush=True)
